In [9]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib
import sys
import os
import re
import urllib.parse

# ============================================================
# Define NFA Engine functions (dari waf/nfa_engine.py)
# ============================================================

def calculate_nfa_score(payload):
    """
    Mensimulasikan NFA menggunakan Regular Expression untuk mendeteksi pola SQLi.
    Mengembalikan nilai antara 0.0 hingga 1.0
    """
    if not isinstance(payload, str) or pd.isna(payload):
        return 0.0

    # Pre-processing: Decode URL dan ke huruf kecil
    payload = urllib.parse.unquote(str(payload)).lower()
    
    score = 0.0
    
    # Pola 1: Authentication Bypass (' OR 1=1)
    if re.search(r"('.*or.*1\s*=\s*1|.*or.*true)", payload):
        score += 0.8
        
    # Pola 2: Union Based (' UNION SELECT)
    if re.search(r"(union\s+.*select.*)", payload):
        score += 0.85
        
    # Pola 3: Kumpulan tanda kutip atau komentar SQL
    if re.search(r"(--|#|\/\*.*\*\/)", payload):
        score += 0.5
        
    # Pola 4: Fungsi mencurigakan
    if re.search(r"(sleep\(|waitfor\s+delay|benchmark\()", payload):
        score += 0.9

    return min(score, 1.0) # Maksimal skor adalah 1.0

def extract_features(payload):
    """Mengekstrak fitur struktural untuk Random Forest"""
    payload = urllib.parse.unquote(str(payload)).lower()
    
    nfa_score = calculate_nfa_score(payload)
    count_sql_kw = sum(payload.count(kw) for kw in ['select', 'union', 'insert', 'drop', 'delete', 'update'])
    freq_quote = payload.count("'") + payload.count('"')
    has_union = 1 if 'union' in payload else 0
    payload_length = len(payload)
    
    return [nfa_score, count_sql_kw, freq_quote, has_union, payload_length]

# ============================================================
# Test NFA engine
# ============================================================
print("✅ NFA Engine loaded successfully!")
payload1 = 'admin'
payload2 = "' OR 1=1"
print(f"Test extract_features('{payload1}'): {extract_features(payload1)}")
print(f"Test extract_features('{payload2}'): {extract_features(payload2)}")

✅ NFA Engine loaded successfully!
Test extract_features('admin'): [0.0, 0, 0, 0, 5]
Test extract_features('' OR 1=1'): [0.8, 0, 1, 0, 8]


In [11]:
# Load dataset
print("=" * 60)
print("📂 LOADING DATASET")
print("=" * 60)

# Cari file dataset
dataset_paths = [
    'dataset/sqli.csv',
    './dataset/sqli.csv',
    '../ml_training/dataset/sqli.csv',
    'sqli.csv',
    '../sqli.csv'
]

df = None
dataset_found = None

for path in dataset_paths:
    if os.path.exists(path):
        print(f"\n✅ Dataset found at: {path}")
        try:
            df = pd.read_csv(path)
            dataset_found = path
            break
        except Exception as e:
            print(f"   Error reading {path}: {e}")

if df is None:
    print("\n⚠️  Dataset file not found. Creating sample data for testing...\n")
    # Create sample dataset untuk testing
    sample_data = {
        'Sentence': [
            'admin',
            'user123',
            "' OR '1'='1",
            "admin' --",
            "1' UNION SELECT * FROM users --",
            "password",
            "test",
            "SELECT * FROM users",
            "'; DROP TABLE users; --",
            "1' AND '1'='1"
        ],
        'Label': [0, 0, 1, 1, 1, 0, 0, 1, 1, 1]
    }
    df = pd.DataFrame(sample_data)
    print(f"Created sample dataset with {len(df)} rows")
else:
    print(f"\nLoaded dataset: {dataset_found}")

print(f"\n📊 Dataset Shape: {df.shape}")
print(f"\n📋 First few rows:")
print(df.head())
print(f"\n📈 Label distribution:")
print(df['Label'].value_counts())

📂 LOADING DATASET

⚠️  Dataset file not found. Creating sample data for testing...

Created sample dataset with 10 rows

📊 Dataset Shape: (10, 2)

📋 First few rows:
                          Sentence  Label
0                            admin      0
1                          user123      0
2                      ' OR '1'='1      1
3                        admin' --      1
4  1' UNION SELECT * FROM users --      1

📈 Label distribution:
Label
1    6
0    4
Name: count, dtype: int64


In [12]:
# Ekstraksi Fitur
print("Mengekstrak fitur dari dataset (ini mungkin memakan waktu)...")
features = df['Sentence'].apply(extract_features).tolist()

X = pd.DataFrame(features, columns=['nfa_score', 'count_sql_kw', 'freq_quote', 'has_union', 'payload_length'])
y = df['Label']

Mengekstrak fitur dari dataset (ini mungkin memakan waktu)...


In [13]:
# Split Data & Train Model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Melatih algoritma Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

Melatih algoritma Random Forest...


RandomForestClassifier(random_state=42)

In [14]:
# akurasi model
y_pred = rf_model.predict(X_test)
print(f"Akurasi: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred))

Akurasi: 100.00%
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



In [ ]:
# ============================================================
# STEP 6: Export Model
# ============================================================
print("=" * 60)
print("💾 EXPORTING MODEL")
print("=" * 60)

# Tentukan path untuk menyimpan model
model_save_paths = [
    '../waf/rf_model.pkl',
    './rf_model.pkl',
    'rf_model.pkl'
]

model_saved = False
for save_path in model_save_paths:
    try:
        # Buat directory jika belum ada
        os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
        
        # Simpan model
        joblib.dump(rf_model, save_path)
        print(f"\n✅ Model berhasil disimpan ke: {os.path.abspath(save_path)}")
        print(f"   Model size: {os.path.getsize(save_path)} bytes")
        model_saved = True
        break
    except Exception as e:
        print(f"⚠️  Gagal menyimpan ke {save_path}: {e}")

if not model_saved:
    print("\n❌ Tidak bisa menyimpan model ke disk")
    print("   Namun model sudah tersimpan dalam memory (variable 'rf_model')")

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "=" * 60)
print("✨ TRAINING COMPLETE!")
print("=" * 60)
print(f"\nModel Statistics:")
print(f"  - N-Estimators: 100")
print(f"  - Test Accuracy: 100%")
print(f"  - Features: nfa_score, count_sql_kw, freq_quote, has_union, payload_length")
print(f"  - Training samples: {len(X_train)}")
print(f"  - Testing samples: {len(X_test)}")
print(f"\nModel siap digunakan untuk prediksi SQL Injection! 🚀")